In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from sklearn.metrics import f1_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./PlantDoc-Split"
BATCH_SIZE = 16 

transforms_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transforms_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(f"{DATA_DIR}/train", transform=transforms_train)
val_data = datasets.ImageFolder(f"{DATA_DIR}/val", transform=transforms_val)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

print("Model indiriliyor...")
model = models.resnet50(weights='IMAGENET1K_V1')
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_data.classes)) 
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001) 

print(f"Eğitim Başlıyor (Cihaz: {DEVICE})...")
for epoch in range(5): 
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"Epoch {epoch+1} - Loss: {running_loss/len(train_loader):.4f} - Val Macro-F1: {f1:.4f}")

torch.save(model.state_dict(), "resnet50_plantdoc.pth")

Model indiriliyor...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/terr0raid/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 34.8MB/s]


Eğitim Başlıyor (Cihaz: cuda)...
Epoch 1 - Loss: 1.9974 - Val Macro-F1: 0.5573
Epoch 2 - Loss: 1.1239 - Val Macro-F1: 0.6822
Epoch 3 - Loss: 0.8491 - Val Macro-F1: 0.6679
Epoch 4 - Loss: 0.6342 - Val Macro-F1: 0.6655
Epoch 5 - Loss: 0.4847 - Val Macro-F1: 0.6493
